#  Data Loading and Preprocessing


In [1]:
# CELL 1: DATA LOADING & PREPROCESSING (MLP + XGBoost)

import json
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings

warnings.filterwarnings('ignore')

# LOAD FILES
print("Loading embeddings...")
emb_data = np.load(r"E:\Projects\CheckThat 2026\my data\merged data embeddings\new_data_dense_embeddings.npz", allow_pickle=True)
embeddings_dict = {k: emb_data[k] for k in emb_data.files}
max_embedding_size = max(np.asarray(arr, dtype=np.float32).reshape(-1).size for arr in embeddings_dict.values())
print(f"Max embedding vector size: {max_embedding_size}")

print("Loading semantic matches...")
with open(r"E:\Projects\CheckThat 2026\my data\semantic_matches.json", 'r', encoding='utf-8') as f:
    semantic_matches = json.load(f)

print("Loading training data...")
with open(r"E:\Projects\CheckThat 2026\task 2\data\train.json", 'r', encoding='utf-8') as f:
    train_data = json.load(f)

# CREATE LOOKUP: claim -> label
claim_to_label = {}
label_map = {'False': 0, 'True': 1, 'Conflicting': 2}
for item in train_data:
    claim = item.get('claim', '').strip()
    label = item.get('label', None)
    if claim:
        claim_to_label[claim] = label

# MATCH & BUILD DATASET
X_list = []
y_list = []
matched_count = 0
skipped_count = 0

for match in semantic_matches:
    new_id = match.get('new_id')
    new_claim = match.get('new_claim', '').strip()
    old_claim = match.get('old_claim', '').strip()

    found_label = None
    if new_claim in claim_to_label:
        found_label = claim_to_label[new_claim]
    elif old_claim in claim_to_label:
        found_label = claim_to_label[old_claim]

    if found_label is None:
        skipped_count += 1
        continue

    emb_key = f"id_{new_id}"
    if emb_key not in embeddings_dict:
        skipped_count += 1
        continue

    embedding_arr = np.asarray(embeddings_dict[emb_key], dtype=np.float32).reshape(-1)

    if embedding_arr.size == 0:
        skipped_count += 1
        continue

    if embedding_arr.size < max_embedding_size:
        embedding_arr = np.pad(embedding_arr, (0, max_embedding_size - embedding_arr.size), mode='constant')
    else:
        embedding_arr = embedding_arr[:max_embedding_size]

    X_list.append(embedding_arr.astype(np.float32))

    if isinstance(found_label, str) and found_label in label_map:
        y_list.append(label_map[found_label])
    else:
        skipped_count += 1
        continue

    matched_count += 1

print(f"✓ Matched: {matched_count} | Skipped: {skipped_count}")

X = np.stack(X_list, axis=0).astype(np.float32)
y = np.asarray(y_list, dtype=np.int32)

print(f"X shape: {X.shape} | y shape: {y.shape}")
print(f"Class distribution: {np.bincount(y)}")

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale for MLP
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✓ Data ready!")

Loading embeddings...
Max embedding vector size: 21504
Loading semantic matches...
Loading training data...
✓ Matched: 6416 | Skipped: 53
X shape: (6416, 21504) | y shape: (6416,)
Class distribution: [3726 1179 1511]
✓ Data ready!


# MLP MODEL


In [2]:
# CELL 2: MLP MODEL

from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

print("Training MLP...")
mlp = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128, 64, 32),  # 5 hidden layers
    activation='relu',
    solver='adam',
    batch_size=32,
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42,
    verbose=1
)

mlp.fit(X_train_scaled, y_train)

# Predict
y_pred_mlp = mlp.predict(X_test_scaled)

# Evaluate
acc = accuracy_score(y_test, y_pred_mlp)
f1 = f1_score(y_test, y_pred_mlp, average='weighted')
prec = precision_score(y_test, y_pred_mlp, average='weighted', zero_division=0)
rec = recall_score(y_test, y_pred_mlp, average='weighted', zero_division=0)

print(f"\n{'='*50}")
print(f"MLP RESULTS")
print(f"{'='*50}")
print(f"Accuracy:  {acc:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"\n{classification_report(y_test, y_pred_mlp, target_names=['False', 'True', 'Conflicting'])}")

# Feature importance (approximation)
print(f"Model trained: {mlp.n_layers_} layers | Parameters: {sum(w.size for w in mlp.coefs_)}")

Training MLP...
Iteration 1, loss = 0.94283264
Validation score: 0.657588
Iteration 2, loss = 0.73262243
Validation score: 0.651751
Iteration 3, loss = 0.61068875
Validation score: 0.661479
Iteration 4, loss = 0.53108154
Validation score: 0.571984
Iteration 5, loss = 0.42269953
Validation score: 0.610895
Iteration 6, loss = 0.32457658
Validation score: 0.651751
Iteration 7, loss = 0.26406723
Validation score: 0.610895
Iteration 8, loss = 0.25355661
Validation score: 0.647860
Iteration 9, loss = 0.17313737
Validation score: 0.630350
Iteration 10, loss = 0.14720939
Validation score: 0.589494
Iteration 11, loss = 0.15593284
Validation score: 0.610895
Iteration 12, loss = 0.12147049
Validation score: 0.647860
Iteration 13, loss = 0.09313243
Validation score: 0.632296
Iteration 14, loss = 0.08998102
Validation score: 0.626459
Iteration 15, loss = 0.11637873
Validation score: 0.579767
Iteration 16, loss = 0.13090031
Validation score: 0.630350
Iteration 17, loss = 0.12377243
Validation score:

# XGBoost MODEL


In [2]:
# CELL 3: XGBOOST MODEL

import pandas as pd
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

print("Training XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=1,
    min_child_weight=1,
    objective='multi:softprob' if len(np.unique(y)) > 2 else 'binary:logistic',
    num_class=3 if len(np.unique(y)) > 2 else None,
    tree_method='hist',
    random_state=42,
    verbosity=1,
    eval_metric='mlogloss' if len(np.unique(y)) > 2 else 'logloss'
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

# Predict
y_pred_xgb = xgb_model.predict(X_test)

# Evaluate
acc = accuracy_score(y_test, y_pred_xgb)
f1 = f1_score(y_test, y_pred_xgb, average='weighted')
prec = precision_score(y_test, y_pred_xgb, average='weighted', zero_division=0)
rec = recall_score(y_test, y_pred_xgb, average='weighted', zero_division=0)

print(f"\n{'='*50}")
print("XGBOOST RESULTS")
print(f"{'='*50}")
print(f"Accuracy:  {acc:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"\n{classification_report(y_test, y_pred_xgb, target_names=['False', 'True', 'Conflicting'])}")

importance_df = pd.DataFrame({
    'feature_index': range(X_train.shape[1]),
    'importance': xgb_model.feature_importances_
}).nlargest(20, 'importance')
print(f"\nTop 20 Important Features:\n{importance_df}")

Training XGBoost...

XGBOOST RESULTS
Accuracy:  0.6324
F1-Score:  0.5935
Precision: 0.5921
Recall:    0.6324

              precision    recall  f1-score   support

       False       0.69      0.89      0.78       746
        True       0.45      0.21      0.29       236
 Conflicting       0.46      0.31      0.37       302

    accuracy                           0.63      1284
   macro avg       0.53      0.47      0.48      1284
weighted avg       0.59      0.63      0.59      1284


Top 20 Important Features:
       feature_index  importance
17264          17264    0.001791
16244          16244    0.001447
1019            1019    0.001403
19482          19482    0.001366
19172          19172    0.001272
1908            1908    0.001157
19312          19312    0.001119
18458          18458    0.001061
15231          15231    0.001060
15137          15137    0.001043
12538          12538    0.001024
4991            4991    0.001012
17310          17310    0.000980
14196          1419

# New MLP Model

In [ ]:
# CAVEMAN ULTRA: Fix Macro F1 & Recall

# PROBLEM: More layers hurt minority classes (True, Conflicting)
# Your True recall: 0.30 → 0.23 (worse!)
# Your Conflicting recall: stuck at 0.30

# SOLUTION: Class imbalance fix

# Get class weights
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = {i: w for i, w in enumerate(class_weights)}
print(f"Class weights: {class_weight_dict}")

# Smaller network + dropout + class weights
mlp_v2 = MLPClassifier(
    hidden_layer_sizes=(512, 256),  # SMALLER, not bigger
    activation='relu',
    solver='adam',
    batch_size=16,  # Smaller batches
    learning_rate='adaptive',
    learning_rate_init=0.0005,  # 2x slower
    max_iter=1000,  # More epochs
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=30,  # Wait longer before stopping
    alpha=0.0001,  # L2 regularization (reduce if still bad)
    random_state=42,
    verbose=1
)

mlp_v2.fit(X_train_scaled, y_train)

y_pred = mlp_v2.predict(X_test_scaled)

from sklearn.metrics import classification_report, f1_score, recall_score

print("\n" + "="*60)
print("MLP V2 - MACRO F1 & RECALL FOCUS")
print("="*60)
print(f"Macro F1:     {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"Macro Recall: {recall_score(y_test, y_pred, average='macro'):.4f}")
print(f"\n{classification_report(y_test, y_pred, target_names=['False', 'True', 'Conflicting'])}")

Class weights: {0: 0.5740492170022371, 1: 1.8140685754683634, 2: 1.4149434794596085}
Iteration 1, loss = 1.88112082
Validation score: 0.581712
Iteration 2, loss = 0.68139010
Validation score: 0.610895
Iteration 3, loss = 0.37403747
Validation score: 0.591440
Iteration 4, loss = 0.22650534
Validation score: 0.618677
Iteration 5, loss = 0.15257435
Validation score: 0.581712
Iteration 6, loss = 0.14470628
Validation score: 0.597276
Iteration 7, loss = 0.21472294
Validation score: 0.595331
Iteration 8, loss = 0.25529692
Validation score: 0.583658
Iteration 9, loss = 0.23010099
Validation score: 0.587549
Iteration 10, loss = 0.15891233
Validation score: 0.589494
Iteration 11, loss = 0.17169947
Validation score: 0.599222
Iteration 12, loss = 0.13646799
Validation score: 0.595331
Iteration 13, loss = 0.13980846
Validation score: 0.587549
Iteration 14, loss = 0.18044616
Validation score: 0.564202
Iteration 15, loss = 0.19237395
Validation score: 0.583658
Iteration 16, loss = 0.15214856
Validat

# GMM Model

In [ ]:
# GMM CLUSTERING WITH EXISTING PREPROCESSING

from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    confusion_matrix, precision_score, recall_score, f1_score,
    classification_report, accuracy_score
)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Training GMM with 3 clusters...")
gmm = GaussianMixture(
    n_components=3,
    covariance_type='full',  # Full covariance matrix
    max_iter=200,
    random_state=42,
    verbose=1
)

# Fit on training data
gmm.fit(X_train_scaled)

# Predict clusters on TEST data
y_pred_gmm = gmm.predict(X_test_scaled)

print(f"\n✓ GMM trained successfully!")
print(f"Cluster distribution in test set: {np.bincount(y_pred_gmm)}")

# CONFUSION MATRIX
cm = confusion_matrix(y_test, y_pred_gmm)
print("\n" + "="*70)
print("CONFUSION MATRIX")
print("="*70)
print(cm)
print()

# DETAILED METRICS
print("="*70)
print("DETAILED METRICS PER CLASS")
print("="*70)
prec = precision_score(y_test, y_pred_gmm, average=None, zero_division=0)
rec = recall_score(y_test, y_pred_gmm, average=None, zero_division=0)
f1 = f1_score(y_test, y_pred_gmm, average=None, zero_division=0)
support = np.bincount(y_test)

class_names = ['False', 'True', 'Conflicting']

metrics_df = pd.DataFrame({
    'Class': class_names,
    'Precision': prec,
    'Recall': rec,
    'F1-Score': f1,
    'Support': support
})

print(metrics_df.to_string(index=False))

# AVERAGES
print("\n" + "="*70)
print("AVERAGES")
print("="*70)
prec_macro = precision_score(y_test, y_pred_gmm, average='macro', zero_division=0)
rec_macro = recall_score(y_test, y_pred_gmm, average='macro', zero_division=0)
f1_macro = f1_score(y_test, y_pred_gmm, average='macro', zero_division=0)

prec_weighted = precision_score(y_test, y_pred_gmm, average='weighted', zero_division=0)
rec_weighted = recall_score(y_test, y_pred_gmm, average='weighted', zero_division=0)
f1_weighted = f1_score(y_test, y_pred_gmm, average='weighted', zero_division=0)

accuracy = accuracy_score(y_test, y_pred_gmm)

avg_df = pd.DataFrame({
    'Metric': ['Macro Avg', 'Weighted Avg', 'Overall Accuracy'],
    'Precision': [prec_macro, prec_weighted, accuracy],
    'Recall': [rec_macro, rec_weighted, '-'],
    'F1-Score': [f1_macro, f1_weighted, '-']
})

print(avg_df.to_string(index=False))

# FULL CLASSIFICATION REPORT
print("\n" + "="*70)
print("FULL CLASSIFICATION REPORT")
print("="*70)
print(classification_report(
    y_test, y_pred_gmm,
    target_names=class_names,
    zero_division=0
))

# CONFUSION MATRIX HEATMAP
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('GMM Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('True Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.show()

# GMM INFO
print("\n" + "="*70)
print("GMM MODEL INFO")
print("="*70)
print(f"BIC Score: {gmm.bic(X_test_scaled):.2f}")
print(f"AIC Score: {gmm.aic(X_test_scaled):.2f}")
print(f"Log-Likelihood: {gmm.score(X_test_scaled):.4f}")